# 03. 평가 지표 (Evaluation Metrics)

## 학습 목표
- 분류 지표 (Accuracy, Precision, Recall, F1)를 직접 구현하고 의미 이해
- Confusion Matrix, ROC Curve, AUC 시각화
- 회귀 지표 (MSE, RMSE, R², MAE) 이해
- Cross-Validation을 직접 구현
- 학습곡선으로 Overfitting/Underfitting 진단

## 참고 자료
- [StatQuest - Confusion Matrix](https://www.youtube.com/watch?v=Kdsp6soqA7o)
- [StatQuest - ROC and AUC](https://www.youtube.com/watch?v=4jRBRDbJemM)

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.datasets import make_classification, load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, auc,
    mean_squared_error, mean_absolute_error, r2_score
)

np.random.seed(42)

## 1. 분류 지표: Accuracy, Precision, Recall, F1-Score

### 왜 Accuracy만으로는 부족한가?

예시: 암 진단 데이터에서 실제 양성이 1%뿐이라면,
"모두 음성"이라고 예측해도 Accuracy = 99%.
하지만 이 모델은 **쓸모없다**.

### 각 지표의 의미

|  | 예측 Positive | 예측 Negative |
|--|:---:|:---:|
| **실제 Positive** | TP (True Positive) | FN (False Negative) |
| **실제 Negative** | FP (False Positive) | TN (True Negative) |

$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$

$$\text{Precision} = \frac{TP}{TP + FP} \quad \text{(양성 예측 중 실제 양성 비율)}$$

$$\text{Recall} = \frac{TP}{TP + FN} \quad \text{(실제 양성 중 찾아낸 비율)}$$

$$\text{F1} = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}} \quad \text{(조화 평균)}$$

### 1.1 직접 구현

In [ ]:
def my_accuracy(y_true, y_pred):
    """정확도: 전체 중 맞은 비율"""
    return np.sum(y_true == y_pred) / len(y_true)

def my_precision(y_true, y_pred):
    """정밀도: 양성 예측 중 실제 양성"""
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    return tp / (tp + fp) if (tp + fp) > 0 else 0.0

def my_recall(y_true, y_pred):
    """재현율: 실제 양성 중 찾아낸 비율"""
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fn = np.sum((y_pred == 0) & (y_true == 1))
    return tp / (tp + fn) if (tp + fn) > 0 else 0.0

def my_f1(y_true, y_pred):
    """F1 Score: Precision과 Recall의 조화 평균"""
    p = my_precision(y_true, y_pred)
    r = my_recall(y_true, y_pred)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0

# 테스트 데이터
y_true = np.array([1, 1, 1, 1, 0, 0, 0, 0, 1, 0])
y_pred = np.array([1, 1, 0, 1, 0, 1, 0, 0, 0, 0])

print("직접 구현 vs sklearn:")
print(f"  Accuracy:  {my_accuracy(y_true, y_pred):.4f} | {accuracy_score(y_true, y_pred):.4f}")
print(f"  Precision: {my_precision(y_true, y_pred):.4f} | {precision_score(y_true, y_pred):.4f}")
print(f"  Recall:    {my_recall(y_true, y_pred):.4f} | {recall_score(y_true, y_pred):.4f}")
print(f"  F1 Score:  {my_f1(y_true, y_pred):.4f} | {f1_score(y_true, y_pred):.4f}")

print("\n구체적으로:")
tp = np.sum((y_pred == 1) & (y_true == 1))
fp = np.sum((y_pred == 1) & (y_true == 0))
fn = np.sum((y_pred == 0) & (y_true == 1))
tn = np.sum((y_pred == 0) & (y_true == 0))
print(f"  TP={tp}, FP={fp}, FN={fn}, TN={tn}")
print(f"  Precision = {tp}/({tp}+{fp}) = {tp/(tp+fp):.4f}")
print(f"  Recall    = {tp}/({tp}+{fn}) = {tp/(tp+fn):.4f}")

### 1.2 불균형 데이터에서의 지표 비교

In [ ]:
# 불균형 데이터 생성 (양성 5%, 음성 95%)
np.random.seed(42)
X_imb, y_imb = make_classification(n_samples=1000, n_features=10,
                                    weights=[0.95, 0.05], random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X_imb, y_imb, test_size=0.3, random_state=42)

print(f"전체 데이터 - 양성: {y_imb.sum()}, 음성: {len(y_imb) - y_imb.sum()}")
print(f"양성 비율: {y_imb.mean():.2%}")
print()

# 모델 1: "모두 음성"으로 예측
y_pred_all_neg = np.zeros_like(y_test)

# 모델 2: 로지스틱 회귀
clf = LogisticRegression(random_state=42, max_iter=1000)
clf.fit(X_train, y_train)
y_pred_lr = clf.predict(X_test)

print(f"{'지표':<15} {'모두 음성 예측':>15} {'로지스틱 회귀':>15}")
print("-" * 48)
print(f"{'Accuracy':<15} {accuracy_score(y_test, y_pred_all_neg):>15.4f} {accuracy_score(y_test, y_pred_lr):>15.4f}")
print(f"{'Precision':<15} {precision_score(y_test, y_pred_all_neg, zero_division=0):>15.4f} {precision_score(y_test, y_pred_lr):>15.4f}")
print(f"{'Recall':<15} {recall_score(y_test, y_pred_all_neg):>15.4f} {recall_score(y_test, y_pred_lr):>15.4f}")
print(f"{'F1 Score':<15} {f1_score(y_test, y_pred_all_neg):>15.4f} {f1_score(y_test, y_pred_lr):>15.4f}")

print("\n→ '모두 음성'도 Accuracy는 높지만, Recall=0 (양성을 하나도 못 찾음)")
print("→ 불균형 데이터에서는 F1 Score가 더 신뢰할 만한 지표")

---
## 2. Confusion Matrix 시각화

분류 결과를 한눈에 보여주는 표. 각 칸이 TP, FP, FN, TN을 나타냄.

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title='Confusion Matrix', ax=None):
    """Confusion Matrix를 heatmap으로 시각화"""
    cm = confusion_matrix(y_true, y_pred)
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 5))
    
    im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
    ax.figure.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    
    labels = ['Negative (0)', 'Positive (1)']
    ax.set(xticks=[0, 1], yticks=[0, 1],
           xticklabels=labels, yticklabels=labels,
           ylabel='Actual', xlabel='Predicted',
           title=title)
    
    # 각 칸에 숫자와 레이블 표시
    cell_labels = [['TN', 'FP'], ['FN', 'TP']]
    thresh = cm.max() / 2.
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cell_labels[i][j]}\n{cm[i, j]}",
                    ha="center", va="center", fontsize=14, fontweight='bold',
                    color="white" if cm[i, j] > thresh else "black")
    return cm

# 유방암 데이터로 실제 예시
data = load_breast_cancer()
X_bc, y_bc = data.data, data.target
X_train_bc, X_test_bc, y_train_bc, y_test_bc = train_test_split(
    X_bc, y_bc, test_size=0.3, random_state=42)

clf_bc = LogisticRegression(max_iter=10000, random_state=42)
clf_bc.fit(X_train_bc, y_train_bc)
y_pred_bc = clf_bc.predict(X_test_bc)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 왼쪽: Confusion Matrix
cm = plot_confusion_matrix(y_test_bc, y_pred_bc, 'Breast Cancer Classification', ax=axes[0])

# 오른쪽: 지표 요약 바 차트
ax = axes[1]
metrics = {
    'Accuracy': accuracy_score(y_test_bc, y_pred_bc),
    'Precision': precision_score(y_test_bc, y_pred_bc),
    'Recall': recall_score(y_test_bc, y_pred_bc),
    'F1 Score': f1_score(y_test_bc, y_pred_bc),
}
colors = ['steelblue', 'green', 'orange', 'red']
bars = ax.bar(metrics.keys(), metrics.values(), color=colors, alpha=0.8)
for bar, val in zip(bars, metrics.values()):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontweight='bold')
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Classification Metrics')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

---
## 3. ROC Curve와 AUC

### ROC Curve (Receiver Operating Characteristic)

분류 threshold를 0에서 1로 변화시키면서 **TPR(Recall)** vs **FPR**을 그린 곡선.

$$\text{TPR (True Positive Rate)} = \frac{TP}{TP + FN} = \text{Recall}$$

$$\text{FPR (False Positive Rate)} = \frac{FP}{FP + TN}$$

### AUC (Area Under Curve)

ROC 곡선 아래 면적. 1에 가까울수록 좋은 모델.
- AUC = 1.0: 완벽한 분류
- AUC = 0.5: 랜덤 분류 (쓸모없음)
- AUC < 0.5: 예측을 뒤집으면 더 나음

### 3.1 ROC Curve 직접 그리기

In [ ]:
# 예측 확률 얻기
y_scores = clf_bc.predict_proba(X_test_bc)[:, 1]  # 양성 확률

# 직접 ROC curve 계산
def my_roc_curve(y_true, y_scores, n_thresholds=200):
    """ROC curve를 직접 계산"""
    thresholds = np.linspace(1, 0, n_thresholds)
    tpr_list = []
    fpr_list = []
    
    for threshold in thresholds:
        y_pred = (y_scores >= threshold).astype(int)
        
        tp = np.sum((y_pred == 1) & (y_true == 1))
        fp = np.sum((y_pred == 1) & (y_true == 0))
        fn = np.sum((y_pred == 0) & (y_true == 1))
        tn = np.sum((y_pred == 0) & (y_true == 0))
        
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
        
        tpr_list.append(tpr)
        fpr_list.append(fpr)
    
    return np.array(fpr_list), np.array(tpr_list), thresholds

# 직접 계산
fpr_manual, tpr_manual, thresholds_manual = my_roc_curve(y_test_bc, y_scores)
auc_manual = np.trapz(tpr_manual, fpr_manual)  # 사다리꼴 적분

# sklearn
fpr_sk, tpr_sk, thresholds_sk = roc_curve(y_test_bc, y_scores)
auc_sk = auc(fpr_sk, tpr_sk)

print(f"AUC (직접 계산): {auc_manual:.4f}")
print(f"AUC (sklearn):   {auc_sk:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 왼쪽: ROC Curve
ax = axes[0]
ax.plot(fpr_manual, tpr_manual, 'b-', linewidth=2, label=f'직접 구현 (AUC={auc_manual:.3f})')
ax.plot(fpr_sk, tpr_sk, 'r--', linewidth=2, label=f'sklearn (AUC={auc_sk:.3f})')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random (AUC=0.5)')
ax.fill_between(fpr_manual, tpr_manual, alpha=0.1, color='blue')
ax.set_xlabel('FPR (False Positive Rate)')
ax.set_ylabel('TPR (True Positive Rate = Recall)')
ax.set_title('ROC Curve')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')

# 오른쪽: Threshold에 따른 Precision, Recall, F1 변화
ax = axes[1]
thresholds_plot = np.linspace(0.01, 0.99, 100)
precisions = []
recalls = []
f1s = []

for t in thresholds_plot:
    y_pred_t = (y_scores >= t).astype(int)
    precisions.append(precision_score(y_test_bc, y_pred_t, zero_division=0))
    recalls.append(recall_score(y_test_bc, y_pred_t))
    f1s.append(f1_score(y_test_bc, y_pred_t))

ax.plot(thresholds_plot, precisions, 'g-', linewidth=2, label='Precision')
ax.plot(thresholds_plot, recalls, 'b-', linewidth=2, label='Recall')
ax.plot(thresholds_plot, f1s, 'r-', linewidth=2, label='F1 Score')

best_t_idx = np.argmax(f1s)
ax.axvline(x=thresholds_plot[best_t_idx], color='gray', linestyle='--', alpha=0.5,
           label=f'Best threshold={thresholds_plot[best_t_idx]:.2f}')

ax.set_xlabel('Threshold')
ax.set_ylabel('Score')
ax.set_title('Threshold에 따른 지표 변화')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nF1이 최대인 threshold: {thresholds_plot[best_t_idx]:.2f}")
print(f"  Precision: {precisions[best_t_idx]:.4f}")
print(f"  Recall:    {recalls[best_t_idx]:.4f}")
print(f"  F1:        {f1s[best_t_idx]:.4f}")

---
## 4. 회귀 지표: MSE, RMSE, R², MAE

| 지표 | 수식 | 의미 |
|------|------|------|
| MSE | $\frac{1}{n}\sum(y - \hat{y})^2$ | 평균 제곱 오차 |
| RMSE | $\sqrt{\text{MSE}}$ | 원래 단위로 해석 가능 |
| MAE | $\frac{1}{n}\sum|y - \hat{y}|$ | 평균 절대 오차, 이상치에 강건 |
| R² | $1 - \frac{\sum(y - \hat{y})^2}{\sum(y - \bar{y})^2}$ | 설명력 (1에 가까울수록 좋음) |

In [ ]:
# 회귀 지표 직접 구현
def my_mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def my_rmse(y_true, y_pred):
    return np.sqrt(my_mse(y_true, y_pred))

def my_mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def my_r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)    # 잔차 제곱합
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)  # 총 제곱합
    return 1 - ss_res / ss_tot

# 테스트 데이터
np.random.seed(42)
X_reg = np.linspace(0, 10, 50).reshape(-1, 1)
y_true_reg = 3 * X_reg.ravel() + 7
y_noisy = y_true_reg + np.random.randn(50) * 3

from sklearn.linear_model import LinearRegression
model_reg = LinearRegression()
model_reg.fit(X_reg, y_noisy)
y_pred_reg = model_reg.predict(X_reg)

print("직접 구현 vs sklearn:")
print(f"  MSE:  {my_mse(y_noisy, y_pred_reg):.4f} | {mean_squared_error(y_noisy, y_pred_reg):.4f}")
print(f"  RMSE: {my_rmse(y_noisy, y_pred_reg):.4f} | {np.sqrt(mean_squared_error(y_noisy, y_pred_reg)):.4f}")
print(f"  MAE:  {my_mae(y_noisy, y_pred_reg):.4f} | {mean_absolute_error(y_noisy, y_pred_reg):.4f}")
print(f"  R²:   {my_r2(y_noisy, y_pred_reg):.4f} | {r2_score(y_noisy, y_pred_reg):.4f}")

In [ ]:
# R² 직관적 이해 시각화
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

scenarios = [
    ('Perfect Fit', y_true_reg, y_true_reg),
    ('Good Fit (R²>0)', y_noisy, y_pred_reg),
    ('Mean Prediction (R²=0)', y_noisy, np.full_like(y_noisy, y_noisy.mean())),
]

for ax, (title, y_actual, y_fitted) in zip(axes, scenarios):
    r2 = my_r2(y_actual, y_fitted)
    
    ax.scatter(X_reg, y_actual, alpha=0.6, s=20, label='Data')
    ax.plot(X_reg, y_fitted, 'r-', linewidth=2, label=f'Prediction')
    
    # 잔차 표시 (몇 개만)
    for i in range(0, len(X_reg), 5):
        ax.plot([X_reg[i], X_reg[i]], [y_actual[i], y_fitted[i]],
                'g-', alpha=0.3, linewidth=1)
    
    ax.set_xlabel('X')
    ax.set_ylabel('y')
    ax.set_title(f'{title}\nR² = {r2:.4f}')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("R² 해석:")
print("  R² = 1.0: 모델이 데이터의 변동을 100% 설명")
print("  R² = 0.0: 모델이 평균값 예측만큼만 좋음 (전혀 안 좋음)")
print("  R² < 0.0: 모델이 평균값 예측보다도 못함")

---
## 5. Cross-Validation (교차 검증)

### 왜 필요한가?

Train/Test를 한 번만 나누면 **어떻게 나누느냐에 따라 결과가 달라짐**.
K-Fold Cross-Validation은 데이터를 K번 다르게 나눠서 평균 성능을 측정.

```
Fold 1: [TEST] [TRAIN] [TRAIN] [TRAIN] [TRAIN]
Fold 2: [TRAIN] [TEST] [TRAIN] [TRAIN] [TRAIN]
Fold 3: [TRAIN] [TRAIN] [TEST] [TRAIN] [TRAIN]
Fold 4: [TRAIN] [TRAIN] [TRAIN] [TEST] [TRAIN]
Fold 5: [TRAIN] [TRAIN] [TRAIN] [TRAIN] [TEST]
```

### 5.1 K-Fold Cross-Validation 직접 구현

In [ ]:
def my_kfold_cv(X, y, model_class, k=5, **model_kwargs):
    """K-Fold Cross-Validation 직접 구현"""
    n = len(X)
    fold_size = n // k
    indices = np.arange(n)
    np.random.shuffle(indices)
    
    scores = []
    
    for i in range(k):
        # 테스트 인덱스
        test_start = i * fold_size
        test_end = test_start + fold_size if i < k - 1 else n
        test_idx = indices[test_start:test_end]
        train_idx = np.concatenate([indices[:test_start], indices[test_end:]])
        
        X_train_fold, X_test_fold = X[train_idx], X[test_idx]
        y_train_fold, y_test_fold = y[train_idx], y[test_idx]
        
        # 모델 학습
        model = model_class(**model_kwargs)
        model.fit(X_train_fold, y_train_fold)
        score = model.score(X_test_fold, y_test_fold)
        scores.append(score)
        print(f"  Fold {i+1}: score = {score:.4f}")
    
    return np.array(scores)

# 유방암 데이터로 테스트
print("직접 구현한 5-Fold CV:")
np.random.seed(42)
scores_manual = my_kfold_cv(X_bc, y_bc, LogisticRegression, k=5, max_iter=10000, random_state=42)
print(f"  평균: {scores_manual.mean():.4f} (+/- {scores_manual.std():.4f})")

print("\nsklearn cross_val_score:")
scores_sk = cross_val_score(LogisticRegression(max_iter=10000, random_state=42),
                             X_bc, y_bc, cv=5, scoring='accuracy')
for i, s in enumerate(scores_sk):
    print(f"  Fold {i+1}: score = {s:.4f}")
print(f"  평균: {scores_sk.mean():.4f} (+/- {scores_sk.std():.4f})")

### 5.2 K값에 따른 CV 결과 변화

In [ ]:
k_values = [2, 3, 5, 10, 20]
results = {}

for k in k_values:
    scores = cross_val_score(LogisticRegression(max_iter=10000, random_state=42),
                              X_bc, y_bc, cv=k, scoring='accuracy')
    results[k] = scores

fig, ax = plt.subplots(figsize=(10, 5))
positions = range(len(k_values))
bp = ax.boxplot([results[k] for k in k_values], positions=positions, widths=0.6)
ax.set_xticks(positions)
ax.set_xticklabels([f'K={k}\n(mean={results[k].mean():.3f})' for k in k_values])
ax.set_ylabel('Accuracy')
ax.set_title('K-Fold CV: K값에 따른 성능 분포')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nK가 클수록:")
print("  + 학습 데이터가 더 많아져 bias 감소")
print("  + 더 안정적인 추정")
print("  - 계산 비용 증가")
print("  - fold 간 학습 데이터 중복 증가 → variance 약간 증가")
print("\n일반적으로 K=5 또는 K=10이 가장 많이 사용됨")

---
## 6. Overfitting vs Underfitting: 학습곡선 (Learning Curve)

학습 데이터 크기를 점점 늘려가면서 **Train/Validation 성능**을 관찰.

- **Underfitting**: Train과 Validation 모두 낮은 성능, 가까이 수렴
- **Good Fit**: Train 높고, Validation도 점차 상승하여 Train에 가까워짐
- **Overfitting**: Train은 높지만, Validation과의 큰 갭 (gap)

In [ ]:
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

# 학습곡선 시각화
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

models = [
    ('Underfitting\n(LogReg, 비선형 데이터)', LogisticRegression(max_iter=10000)),
    ('Good Fit\n(SVM, RBF kernel)', SVC(kernel='rbf', gamma='scale')),
    ('Overfitting\n(Decision Tree, no limit)', DecisionTreeClassifier(random_state=42)),
]

# 비선형 분류 데이터 생성
from sklearn.datasets import make_moons
X_moon, y_moon = make_moons(n_samples=300, noise=0.3, random_state=42)

for ax, (title, model) in zip(axes, models):
    train_sizes, train_scores, val_scores = learning_curve(
        model, X_moon, y_moon, train_sizes=np.linspace(0.1, 1.0, 10),
        cv=5, scoring='accuracy', random_state=42
    )
    
    train_mean = train_scores.mean(axis=1)
    train_std = train_scores.std(axis=1)
    val_mean = val_scores.mean(axis=1)
    val_std = val_scores.std(axis=1)
    
    ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color='blue')
    ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.1, color='red')
    ax.plot(train_sizes, train_mean, 'b-o', markersize=4, label=f'Train ({train_mean[-1]:.3f})')
    ax.plot(train_sizes, val_mean, 'r-o', markersize=4, label=f'Validation ({val_mean[-1]:.3f})')
    
    gap = train_mean[-1] - val_mean[-1]
    ax.set_xlabel('Training Set Size')
    ax.set_ylabel('Accuracy')
    ax.set_title(f'{title}\nGap: {gap:.3f}')
    ax.legend(loc='lower right', fontsize=8)
    ax.set_ylim(0.5, 1.05)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("학습곡선 해석:")
print("  Underfitting: Train/Val 모두 낮음 → 더 복잡한 모델 필요")
print("  Good Fit:     Train 높고, Val도 근접 → 적절한 모델")
print("  Overfitting:  Train=1.0, Val 낮음 (큰 gap) → 정규화 또는 더 많은 데이터 필요")

### 6.1 Overfitting 해결 방법: 학습곡선으로 확인

In [ ]:
# Decision Tree의 max_depth를 제한하여 overfitting 완화
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

depths = [None, 5, 2]
depth_labels = ['No Limit (Overfitting)', 'max_depth=5 (Good)', 'max_depth=2 (Underfitting)']

for ax, depth, label in zip(axes, depths, depth_labels):
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    
    train_sizes, train_scores, val_scores = learning_curve(
        model, X_moon, y_moon, train_sizes=np.linspace(0.1, 1.0, 10),
        cv=5, scoring='accuracy', random_state=42
    )
    
    train_mean = train_scores.mean(axis=1)
    val_mean = val_scores.mean(axis=1)
    
    ax.fill_between(train_sizes, train_scores.mean(axis=1) - train_scores.std(axis=1),
                     train_scores.mean(axis=1) + train_scores.std(axis=1), alpha=0.1, color='blue')
    ax.fill_between(train_sizes, val_scores.mean(axis=1) - val_scores.std(axis=1),
                     val_scores.mean(axis=1) + val_scores.std(axis=1), alpha=0.1, color='red')
    ax.plot(train_sizes, train_mean, 'b-o', markersize=4, label=f'Train ({train_mean[-1]:.3f})')
    ax.plot(train_sizes, val_mean, 'r-o', markersize=4, label=f'Val ({val_mean[-1]:.3f})')
    
    ax.set_xlabel('Training Set Size')
    ax.set_ylabel('Accuracy')
    ax.set_title(f'Decision Tree: {label}')
    ax.legend(loc='lower right', fontsize=8)
    ax.set_ylim(0.5, 1.05)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 연습 문제

아래 문제를 직접 풀어보세요.

### 연습 1: 다중 클래스 분류 평가

Iris 데이터셋 (3클래스)에 대해 다중 클래스 confusion matrix를 그리고,
각 클래스별 precision, recall, f1-score를 계산하세요.

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()
X_iris, y_iris = iris.data, iris.target

# TODO:
#   1. train/test 분할 (test_size=0.3)
#   2. LogisticRegression으로 학습
#   3. 3x3 confusion matrix 시각화 (heatmap)
#      - 축 레이블에 클래스 이름 (setosa, versicolor, virginica) 표시
#   4. sklearn.metrics.classification_report로 클래스별 지표 출력


### 연습 2: 여러 모델의 ROC Curve 비교

같은 데이터셋에 대해 여러 모델의 ROC Curve와 AUC를 비교하세요.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# 유방암 데이터 사용
X_train_bc, X_test_bc, y_train_bc, y_test_bc = train_test_split(
    X_bc, y_bc, test_size=0.3, random_state=42)

# TODO:
#   1. 아래 4개 모델을 학습:
#      - LogisticRegression
#      - DecisionTreeClassifier
#      - KNeighborsClassifier(n_neighbors=5)
#      - SVC(probability=True)
#   2. 각 모델의 predict_proba로 확률 예측
#   3. 한 그래프에 4개의 ROC Curve + AUC 표시
#   4. 어떤 모델이 가장 좋은지 판단


---
## 핵심 정리

| 개념 | 핵심 내용 | 언제 사용 |
|------|-----------|----------|
| Accuracy | 전체 정확도 | 균형 잡힌 데이터셋 |
| Precision | 양성 예측의 정확도 | 거짓 양성 비용이 클 때 (스팸 필터) |
| Recall | 실제 양성 탐지율 | 거짓 음성 비용이 클 때 (암 진단) |
| F1 Score | Precision-Recall 조화 평균 | 불균형 데이터, 종합 평가 |
| ROC/AUC | threshold 무관 성능 | 모델 간 비교, threshold 선택 |
| MSE/RMSE | 제곱 오차 | 회귀, 큰 오차에 민감 |
| R² | 설명력 | 회귀 모델의 전반적 성능 |
| Cross-Validation | K번 반복 평가 | 모든 모델 평가에 필수 |
| Learning Curve | 데이터 크기별 성능 | Overfitting/Underfitting 진단 |

**다음 노트북**: [../02-neural-networks/01-perceptron-to-mlp.ipynb](../02-neural-networks/01-perceptron-to-mlp.ipynb) - 신경망 기초